# King County trên GPU (Kaggle)

Trên CPU, FT-Transformer mất ~46 s/epoch và TabNet ~6 s/epoch với King County, nên phần tuning của dataset này chạy ở đây.

**Chuẩn bị**
1. Nén thư mục `dl/` (gồm cả `splits/`) thành `dl.zip`, tạo Kaggle Dataset tên `house-dl-code` từ file đó.
2. Tạo notebook mới → *Add Input*: `house-dl-code`, `harlfoxem/housesalesprediction`, `yasserh/housing-prices-dataset`.
3. *Settings*: Accelerator = **GPU T4/P100**, Internet = **On**.
4. Một session tối đa 12 giờ: chạy FT-Transformer và TabNet ở **2 session riêng** (hoặc 2 notebook), dùng *Save Version → Save & Run All* để chạy nền. `--timeout` dừng tuning trước khi hết giờ; study lưu trong `optuna.db` nên có thể chạy tiếp ở session sau (upload lại thư mục `results/`).

In [ ]:
import glob, os, shutil

# code: /kaggle/input/house-dl-code/dl (Kaggle tự giải nén zip)
src = glob.glob('/kaggle/input/house-dl-code/**/run.py', recursive=True)[0]
shutil.copytree(os.path.dirname(src), '/kaggle/working/dl', dirs_exist_ok=True)

# data: gom 2 CSV vào một thư mục
os.makedirs('/kaggle/working/data', exist_ok=True)
for name in ('kc_house_data.csv', 'Housing.csv'):
    shutil.copy(glob.glob(f'/kaggle/input/**/{name}', recursive=True)[0], '/kaggle/working/data/')
os.environ['HOUSE_DATA_DIR'] = '/kaggle/working/data'
%cd /kaggle/working/dl

In [ ]:
!pip install -q pytorch-tabnet==4.1.0 rtdl_revisiting_models==0.0.2 optuna
!nvidia-smi --query-gpu=name,memory.total --format=csv
# split phải trùng với bản local: lệnh này giữ nguyên file trong splits/ nếu đã có
!python run.py splits
!python -m pytest -q tests/test_no_leakage.py

## Session 1 — FT-Transformer

In [ ]:
!python -u run.py --device cuda tune --dataset kc --model ft_transformer --n-trials 30 --timeout 36000
!python -u run.py --device cuda final --dataset kc --model ft_transformer

## Session 2 — TabNet, ablation, learning curve theo kích thước

In [ ]:
!python -u run.py --device cuda tune --dataset kc --model tabnet --n-trials 30 --timeout 28800
!python -u run.py --device cuda final --dataset kc --model tabnet

In [ ]:
# cần best_params.json của cả 2 model trên kc (copy results/kc/ft_transformer từ session 1 vào đây nếu chạy tách)
!python -u run.py --device cuda size-curve
!python -u run.py --device cuda ablation --dataset kc

In [ ]:
!python run.py summary
!zip -qr /kaggle/working/dl_results_kc.zip results figures
print('Tải /kaggle/working/dl_results_kc.zip về và giải nén vào dl/ ở máy local')